# AtomAlignSPCM3 — interactive analysis

Histograms the counts recorded by `experiments/atom_align_spcm3.py`:

- **A** — SPCM1+SPCM2 reference histogram (atom / no-atom split + threshold)
- **B** — SPCM3 split by atom-present vs atom-absent  (the gap *is* `align_contrast`) — **the alignment figure of merit**
- **C** — SPCM3 vs SPCM1+2 same-window scatter with the Pearson `r` (→ 1 when aligned)

**Which windows are used.** The experiment saves the counts twice: `spcm_all1/2/3` (uncapped — *every* window of the run, archive-only) and `spcm_counts1/2/3` (the applet datasets, trimmed to the last `max_points` shots). The analysis prefers `spcm_all*`, so you get the **whole run**, not just the tail the applet happened to be showing. Runs recorded before `spcm_all*` existed fall back to the rolling datasets and print a truncation warning.

All logic is **imported** from `analyze_align_spcm3.py` — this notebook keeps no copy of it, so the two cannot drift apart. Edit the `.py`, re-run the import cell.

> **Plot backend.** The first code cell picks the backend by what is importable: `widget` (zoom / pan) if `ipympl` is installed, otherwise `inline`. Do **not** hard-code `%matplotlib widget` here — `ipympl` is not installed in `artiq7_claude`, the kernel then falls back to Qt, and `plt.show()` under Qt blocks the kernel (reported as *"The Kernel crashed…"*). For interactive plots: `pip install ipympl ipywidgets`, then restart the kernel.

In [ ]:
# ---------------------------------------------------------------------------
# KERNEL CHECK -- run this FIRST.
#
# matplotlib cannot draw in the `artiq7_claude` conda env: its numpy (1.23.5)
# has a broken BLAS, and matplotlib's Affine2D transforms call np.dot while
# rendering -- so plt.hist() SEGFAULTS and the client reports
# "The Kernel crashed while executing code in the current cell".
# Nothing in this notebook can fix that; the analysis only needs
# numpy / h5py / matplotlib, NOT artiq, so run it on plain Python 3.11:
#
#     VS Code -> kernel picker (top right) -> Python 3.11
#                (C:\Users\CS-Node\AppData\Local\Programs\Python\Python311)
#
# The probe below runs np.dot in a SUBPROCESS, so a broken BLAS kills the child
# process instead of this kernel.
# ---------------------------------------------------------------------------
import subprocess, sys

_blas = subprocess.run(
    [sys.executable, "-c",
     "import numpy as np; np.dot(np.ones((8, 8)), np.ones((8, 8)))"],
    capture_output=True)
if _blas.returncode != 0:
    print("!" * 74)
    print("!! BROKEN BLAS in this kernel -- plotting WILL crash it.")
    print("!!   kernel : {}".format(sys.executable))
    print("!!   fix    : switch the kernel to Python 3.11, then re-run.")
    print("!" * 74)
else:
    print("BLAS ok -- plotting is safe in this kernel.")
    print("kernel:", sys.executable)

# Plotting backend -- picked by what is importable here, never hard-coded.
# A bare `%matplotlib widget` breaks when ipympl is absent: the kernel then
# falls back to a Qt backend, where plt.show() starts a BLOCKING event loop
# (also reported as a "kernel crash"). widget if ipympl is real, else inline.
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
except ImportError:
    get_ipython().run_line_magic("matplotlib", "inline")

import os, importlib
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# Import the analysis module -- single source of truth, no copied code here.
sys.path.insert(0, os.getcwd())
import analyze_align_spcm3 as A
importlib.reload(A)          # picks up edits to the .py without restarting
from analyze_align_spcm3 import (find_run, find_runs, load_run, load_runs,
                                 summarize, analyze)

print("matplotlib backend:", matplotlib.get_backend())
print("results roots:", *[str(r) for r in A.RESULTS_DIRS], sep="\n  ")

## One run

`RID = None` uses the latest run. Set `RID = 2088` (etc.) to pick a specific one.

`analyze()` prints the metrics table, shows the interactive figure, and saves a PNG next to the `.h5` (pass `out_png="..."` to choose the path).

In [ ]:
RID = None   # None = latest run; or e.g. RID = 2088

path = find_run(RID)
fig = analyze(path)

## Several runs pooled

`find_runs()` with no argument lists every `*-AtomAlignSPCM3.h5`; pass RIDs to pick specific ones. Each run's atom-present mask is applied with **its own** `readout_threshold` before pooling.

⚠ Pool only runs taken at the **same SPCM3 position** — otherwise you average over different alignments and the contrast means nothing. The summary warns when pooled runs used different gate widths (raw counts then aren't comparable — compare rates) or different thresholds.

In [ ]:
# The 10 most recent runs on disk:
for p in find_runs()[-10:]:
    print(A._rid_of(p), p)

In [ ]:
RIDS = [2081, 2084, 2088]      # or: find_runs() for every run on disk

fig = analyze(find_runs(RIDS))

## Raw arrays — your own binning

`load_run` / `load_runs` return `(data, meta, rid)`.

- `data`: `c1`, `c2`, `c3`, `sum12`, and the boolean `present` mask
- `meta`: `readout_threshold`, `gate` [s], `n_windows`, plus `source` / `truncated` — which tell you whether you got the full record or only the rolling window

In [ ]:
data, meta, rid = load_run(find_run(RID))
print("RID", rid, "|", meta["source"], "|", meta["n_windows"], "windows")

c3, present = data["c3"], data["present"]
bins = np.arange(c3.min(), c3.max() + 2) - 0.5      # one bin per integer count

plt.figure(figsize=(7, 4))
plt.hist(c3[present], bins=bins, alpha=0.6, label="atom")
plt.hist(c3[~present], bins=bins, alpha=0.6, label="no atom")
plt.xlabel("SPCM3 counts per window")
plt.ylabel("occurrences")
plt.legend()
plt.tight_layout()

In [ ]:
# Metrics only, no figure: prints the table and returns the dict.
res = summarize(find_run(RID))
res